# Основы программирования дискретно-событийных моделей

В ДСМ фундаментальным понятием, на котором построено построено практически всё, является понятие *события* (англ. event).
Светофор переключился с красного сигнала на зелёный, падающее тело ударилось о землю, по кабелю связи пущен электрический импульс и т.п. - это всё примеры событий.

Как мы уже выяснили, при ДСМ события помещаются в *очередь с приоритетом* (англ. priority queue), из которой *в основном цикле моделирования* события извлекаются в порядке времени их наступления.
То есть всегда необходимо знать, **когда** произойдёт каждое событие.

Ещё одним важным понятием, почти всегда сопутствующим событию, является понятие *действия*.
Это то, что происходит при наступлении события.
К примеру, токарь завершил обработку детали (событие) и передал её на следующую ступень технологического процесса (действие), скажем, в цех нанесения покрытий.

В этом цехе покрытий начинается *процесс* нанесения покрытия на деталь.
Этот процесс занимает определённое время, рано или поздно завершаясь.
Завершение процесса - это тоже событие.

Допустим, после покрытия деталь готова к транспортировке в магазин или на склад.
Деталь грузят в грузовой автомобиль, который развозит груз по нескольким точкам.
Начинается процесс доставки, генерирующий несколько событий: автомобиль приехал на склад А, приехал на склад Б и т.д.
Каждому событию отвечают свои действия, заключающиеся в определении того, какие детали должны быть разгружены на текущем складе.

Другими словами, процессы порождают (или *генерируют*) одно или несколько событий, при наступлении которых выполняются соответствующие действия.
Вот и вся суть ДСМ.

На основании этого создадим простой класс события `Event` (на самом деле, он будет простенькой структурой данных), содержащий в себе два поля: время наступления события (`when`) и список действий (`actions`), которые необходимо выполнить при наступлении события.

## Событие

In [10]:
from dataclasses import dataclass, field

@dataclass(order=True, frozen=True)
class Event:
    when: float
    actions: list = field(compare=False)

Подробно о `dataclasses` можно почитать в [документации](https://docs.python.org/3/library/dataclasses.html) стандартной библиотеки.
Достаточно знать, что данный декоратор упрощает запись класса.
В итоге имеет класс `Event` с нужными полями, которые у созданных экземпляров нельзя изменять (`frozen`), а также объекты класса могут быть упорядочены (`order`) по значению поля `when` (времени наступления).
Запись `field(compare=False)` исключает поле `actions` из сравнения.

Далее рассмотрим простой процесс работы светофора.

## Процесс

Рассмотри процесс работы двухцветного светофора.
Вначале горит красный свет в течение времени `dt_red`, затем происходит событие "прошло время `t_red`" - светофор переключается на зелёный и горит в течение `t_green`.
И так далее до заданного времени `until`.

Для наглядности сперва заведём глобальные переменные:

In [11]:
from queue import PriorityQueue
# queue является частью стандартной библиотеки

# Возможные состояния светофора
RED, GREEN = "RED", "GREEN"
# Текущее состояние светофора
current_lights = RED
# Время работы светофора в двух режимах
t_red, t_green = 1, 2
# Текущее модельное время
now = 0
# Очередь событий
events = PriorityQueue()

Применение глобальных переменных в коде чревато трудноуловимыми ошибками.
Не используйте их.
В данном учебнике они применяются исключительно с демонстрационными целями и для упрощения кода.

Сам процесс работы светофора можно описать несложной функцией:

```python
def lights():
    """Функция-процесс работы светофора.
    """
    global events
    # Очередь событий есть глобальная переменная,
    # поэтому для её изменения в области функции
    # необходимо использовать оператор global
    # (иначе не сработает её метод put)

    if current_lights is RED:
        # Добавляем событие "конец красного" в очередь
        events.put(Event(
            when=now + t_red,
            actions=[red2green, lights]
        ))
    else:
        # Добавляем событие "конец зелёного" в очередь
        events.put(Event(
            when=now + t_green,
            actions=[green2red, lights]
        ))
```

In [12]:
def lights():
    """Функция-процесс работы светофора.
    """
    global events
    # Очередь событий есть глобальная переменная,
    # поэтому для её изменения в области функции
    # необходимо использовать оператор global
    # (иначе не сработает её метод put)

    if current_lights is RED:
        # Добавляем событие "конец красного" в очередь
        events.put(Event(
            when=now + t_red,
            actions=[red2green, lights]
        ))
    else:
        # Добавляем событие "конец зелёного" в очередь
        events.put(Event(
            when=now + t_green,
            actions=[green2red, lights]
        ))


def red2green():
    """Переключить светофор с красного на зелёный.
    """
    global current_lights
    # Для возможности изменения глобальной переменной
    # снова пользуемся оператором global

    current_lights = GREEN
    print(f"{now}\t{current_lights}")


def green2red():
    """Переключить светофор с зелёного на красный.
    """
    global current_lights
    current_lights = RED
    print(f"{now}\t{current_lights}")

Вы видите, что при создании событий указаны списки действий, которые необходимо выполнить при наступлении этого самого события.

## Действия

Действия могут быть процессами, т.е. порождать события, при наступлении которых должны выполняться другие действия и т.д.
Это позволяет простыми средствами строить дискретно-событийные модели практически любой сложности.

В рассматриваемой модели светофора возможных действий три:

1. Переключить с красного на зелёный `red2green()`.
2. Переключить с зелёного на красный `green2red()`.
3. Заново запустить генерацию событий вызовом той же самой функции `lights`.

```python
def red2green():
    """Переключить светофор с красного на зелёный.
    """
    global current_lights
    # Для возможности изменения глобальной переменной
    # снова пользуемся оператором global
    current_lights = GREEN
    print(f"{now}\t{current_lights}")
```

```python
def green2red():
    """Переключить светофор с зелёного на красный.
    """
    global current_lights
    current_lights = RED
    print(f"{now}\t{current_lights}")
```

Пока код может казаться непонятным, но мы переходим к основному циклу моделирования, так что вскоре всё прояснится!

## Цикл дискретно-событийного моделирования

Цикл ДСМ достаточно прост (если не сказать тривиален):

1. Запускаем все начальные процессы `processes`.
2. Входим в цикл, условиями выхода из которого является либо опустошение очереди событий `events`, либо то, что следующее событие произойдёт после заданного времени `until`.
   1. Берём из `events` ближайшее `event`.
   2. Обновляем счётчик времени `now` временем наступления `event`.
   3. Выполняем все действия `actions`, соответствующие наступившему событию.

In [13]:
def des_loop(processes, until):
    """Основной цикл дискретно-событийного моделирования.
    """
    global now, events

    # Запускаем все начальные процессы
    for proc in processes:
        proc()
    # В результате в events появятся события
    
    while not events.empty():
        # Берём ближайшее событие
        event = events.get()
        # Проверяем одно из условий выхода из цикла
        if event.when > until:
            break
        # В ином случае обновляем модельное время
        now = event.when
        # Выполняем действия
        for action in event.actions:
            action()

Обратите внимание, вывода сообщений в консоль с помощью `print(...)` в цикле ДСМ нет - это *чистая функция* (функция без побочных эффектов).
Всё необходимое выводится на экран, записывается в файл и т.д. в функциях процессов или действий.

Запустить цикл можно следующим образом:

In [14]:
# Выводим на экран начальное состояние модели
print("Время\tТекущий сигнал")
print(f"{now}\t{current_lights}")

# Формируем список начальных процессов
# (с чего-то же нужно начать?)
processes = [lights]
# Запускаем цикл
des_loop(processes, until=7)

# Далее обрабатывают накопленные данные, строят графики
# и делают прочие полезные действия...

Время	Текущий сигнал
0	RED
1	GREEN
3	RED
4	GREEN
6	RED
7	GREEN


Теперь можно легко понять, почему в процессе `lights()` одним из действий является вызов этой же функции: таким образом получается бесконечный процесс работы светофора.

Таковы основы программной реализации простейшей дискретно-событийной модели.
В следующем разделе поговорим о том, как этот алгоритм может быть улучшен и как использовать возможности Python для упрощения его реализации.

## Исходный код

In [1]:
from dataclasses import dataclass, field
from typing import Callable, List, Union
from queue import PriorityQueue


@dataclass(order=True, frozen=True)
class Event:
    when: Union[int, float]
    actions: List[Callable] = field(compare=False)


def lights():
    global events

    if current_lights is RED:
        events.put(Event(
            when=now + t_red,
            actions=[red2green, lights]
        ))
    else:
        events.put(Event(
            when=now + t_green,
            actions=[green2red, lights]
        ))


def red2green():
    global current_lights
    current_lights = GREEN
    print(f"{now}\t{current_lights}")


def green2red():
    global current_lights
    current_lights = RED
    print(f"{now}\t{current_lights}")


def des_loop(processes, until):
    global now, events

    for proc in processes:
        proc()
    
    while not events.empty():
        event = events.get()
        if event.when > until:
            break

        now = event.when

        for action in event.actions:
            action()


# Возможные состояния светофора
RED, GREEN = "RED", "GREEN"
# Текущее состояние светофора
current_lights = RED
# Время работы светофора в двух режимах
t_red, t_green = 1, 2
# Текущее модельное время
now = 0
# Очередь событий
events = PriorityQueue()
# Моделирование
print("Время\tТекущий сигнал")
print(f"{now}\t{current_lights}")
processes = [lights]
des_loop(processes, until=7)

Время	Текущий сигнал
0	RED
1	GREEN
3	RED
4	GREEN
6	RED
7	GREEN
